In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib as mpl
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from matplotlib import patheffects as pe
from functools import reduce
import sys, pathlib
sys.path.append(str(pathlib.Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/robyns_libraries")))
import Robyn_paper_2_defs
import Robyn_river_floods

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")

connectivity_results_path = base_path / "dphil_paper_2/results/connectivity_results"

output_dir = base_path / "dphil_paper_2/results/figures/connectivity_figures"



In [ ]:
catchments_unionized_final = base_path / "dphil_paper_2/processed_data/major_river_catchments/major_basins_plus_coastal_unionized_final.gpkg"
catchments = gpd.read_file(catchments_unionized_final)
catchments.head()
print("Catchments:", len(catchments))


In [ ]:
jamaica_boundary_path = base_path / "dphil_common_cross_cutting/common_incoming_data/boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(jamaica_boundary.crs)

In [ ]:
# Path to the folder with the CSV
connectivity_results_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers/dphil_paper_2/results/connectivity_results")

# Load the summary CSV
csv_path = connectivity_results_path / "catchment_connectivity_norm_pct_all.csv"
df_all = pd.read_csv(csv_path)


In [ ]:
# --- 3 stacked panels from the single CSV (baseline/partial/long timeframe) ---

# Columns to plot (panel title → column name)
panels = [
    ("Baseline", "norm_scenario_pct_baseline_avg"),
    ("Restoration", "norm_scenario_pct_partial_avg"),
    ("Long timeframe", "norm_scenario_pct_long_timeframe_keep_plantations_avg"),
]

# Build GeoDataFrames for each panel
gdfs = []
for title, col in panels:
    df = df_all[["catchment_uid", col]].copy()
    df = df.rename(columns={col: "norm_scenario_pct"})
    gdf_plot = catchments[["catchment_uid", "geometry"]].merge(df, on="catchment_uid", how="left")
    gdfs.append((title, gdf_plot))

# Figure layout: 3 rows × 2 columns (last col = colorbar spans all rows)
fig = plt.figure(figsize=(8, 12), dpi=300)
gs  = fig.add_gridspec(3, 2, width_ratios=[1, 0.05], height_ratios=[1, 1, 1],
                       wspace=0.06, hspace=0.16)
axes = [fig.add_subplot(gs[0,0]), fig.add_subplot(gs[1,0]), fig.add_subplot(gs[2,0])]
cax  = fig.add_subplot(gs[:, 1])

norm = mpl.colors.Normalize(vmin=0, vmax=100)  # shared 0–100%
cmap = plt.colormaps["Greens"]
letters = ["a)", "b)", "c)"]

for ax, letter, (title, gdf_plot) in zip(axes, letters, gdfs):
    try:
        jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.6, zorder=1)
    except Exception:
        pass

    gdf_plot.plot(
        ax=ax, column="norm_scenario_pct", cmap=cmap, norm=norm,
        linewidth=0.25, edgecolor="black", alpha=0.9, zorder=2,
        missing_kwds={"color":"lightgrey","edgecolor":"white","hatch":"///","label":"No data"}
    )

    reps = gdf_plot.geometry.representative_point()
    for (pt, uid) in zip(reps, gdf_plot["catchment_uid"]):
        if pd.isna(uid):
            continue
        t = ax.text(pt.x, pt.y, str(int(uid)),
                    ha="center", va="center", fontsize=6.5, fontweight="bold",
                    color="black", zorder=3)
        t.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])

    ax.text(0.01, 0.99, letter, transform=ax.transAxes,
            ha="left", va="top", fontsize=12, fontweight="bold",
            path_effects=[pe.withStroke(linewidth=2, foreground="white")])
    ax.set_title(title, fontsize=12, pad=6)
    ax.set_axis_off()

# Shared colorbar
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Normalized connectivity (%)")
cbar.set_ticks([0, 20, 40, 60, 80, 100])

# North arrow + Scale bar on the TOP panel (a) — Robyn_paper_2_defs
try:
    Robyn_paper_2_defs.draw_north_arrow(
        axes[0],
        location=(0.92, 0.92),
        size=0.055,
        fontsize=8,
        label_offset=0.015
    )
    Robyn_paper_2_defs.draw_scale_bar(
        axes[0],
        length_km=20,
        location=(0.92, 0.84),
        linewidth=1,
        tick_height=0.012,
        label_offset=0.018,
        km_offset=0.010
    )
except Exception:
    pass


# Save
panel_name = "fig_5_catchment_level_connectivity_scores"
out_png = Path(output_dir) / f"{panel_name}.png"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
print("Saved panel:", out_png.name, "→", Path(output_dir).resolve())

plt.show()


In [ ]:
baseline_connectivity_lambda_5_path = connectivity_results_path / "catchment_connectivity__baseline__lambda_5.csv"
baseline_connectivity_lambda_5 = gpd.read_file(baseline_connectivity_lambda_5_path)

partial_connectivity_lambda_5_path = connectivity_results_path / "catchment_connectivity__partial__lambda_5.csv"
partial_connectivity_lambda_5 = gpd.read_file(partial_connectivity_lambda_5_path)

reforest_all_connectivity_lambda_5_path = connectivity_results_path / "catchment_connectivity__reforest_all__lambda_5.csv"
reforest_all_connectivity_lambda_5 = gpd.read_file(reforest_all_connectivity_lambda_5_path)

long_timeframe_connectivity_lambda_5_path = connectivity_results_path / "catchment_connectivity__long_timeframe_keep_plantations__lambda_5.csv"
long_timeframe_connectivity_lambda_5 = gpd.read_file(long_timeframe_connectivity_lambda_5_path)


long_timeframe_connectivity_lambda_5.head()


In [ ]:
# # --- Choropleths for 4 scenarios (λ=5), shaded by norm_scenario_pct ---------

# # sanity checks
# assert {'catchment_uid','geometry'}.issubset(catchments.columns), "catchments needs uid + geometry."

# # Scenario → nice title
# scenarios = {
#     "baseline":       "Baseline",
#     "partial":        "Partial",
#     "reforest_all":   "Reforest all",
#     "long_timeframe": "Long timeframe",
# }

# def plot_norm_map(gdf, title, fname, label_ids=True):
#     fig, ax = plt.subplots(figsize=(14, 12), dpi=300)
#     try:
#         jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.6)
#     except Exception:
#         pass

#     norm = mpl.colors.Normalize(vmin=0, vmax=100)
#     cmap = plt.colormaps["Greens"]

#     gdf.plot(ax=ax, column="norm_scenario_pct", cmap=cmap, norm=norm,
#              linewidth=0.25, edgecolor="black", alpha=0.9,
#              missing_kwds={"color":"lightgrey","edgecolor":"white","hatch":"///","label":"No data"})

#     sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
#     cbar = fig.colorbar(sm, ax=ax, fraction=0.030, pad=0.02)
#     cbar.set_label("Normalized connectivity (%)")

#     # ---- LABELS: catchment_uid centered in each polygon with white halo ----
#     if label_ids and "catchment_uid" in gdf.columns:
#         reps = gdf.geometry.representative_point()
#         for (pt, uid) in zip(reps, gdf["catchment_uid"]):
#             if pd.isna(uid):      # skip missing uids
#                 continue
#             t = ax.text(pt.x, pt.y, str(int(uid)),
#                         ha="center", va="center",
#                         fontsize=7, fontweight="bold", color="black", zorder=20)
#             t.set_path_effects([pe.withStroke(linewidth=1.5, foreground="white")])

#     add_scale_bar(ax)
#     add_north_arrow(ax)
#     ax.set_axis_off()
#     ax.set_title(f"{title} — Normalized connectivity (λ=5)", fontsize=16)
#     plt.tight_layout()

#     out_png = Path(output_dir) / f"Figure_{fname}.png"
#     out_pdf = Path(output_dir) / f"Figure_{fname}.pdf"
#     fig.savefig(out_png, dpi=300, bbox_inches="tight")
#     fig.savefig(out_pdf, bbox_inches="tight")
#     print("Saved:", out_png.name, "and", out_pdf.name, "→", Path(output_dir).resolve())
#     plt.show()
    

# for key, nice in scenarios.items():
#     csv_path = connectivity_results_path / f"catchment_connectivity__{key}__lambda_5.csv"
#     if not csv_path.exists():
#         print(f"WARNING: missing CSV → {csv_path}")
#         continue

#     # read attributes (CSV) and join to polygons (GeoDataFrame)
#     df = pd.read_csv(csv_path, usecols=["catchment_uid", "norm_scenario_pct"])
#     gdf_plot = catchments[["catchment_uid","geometry"]].merge(df, on="catchment_uid", how="left")

#     plot_norm_map(gdf_plot, f"{nice}", f"{key}_normalized_connectivity_lambda5", label_ids=True)

In [ ]:
# # --- 2x2 panel with right-side colorbar + panel letters ---------------------
# # (Assumes: catchments, jamaica_boundary, connectivity_results_path, output_dir exist)

# # Load & merge attributes for each scenario (λ=5)
# gdfs = []
# for key, nice in (scenarios.items() if isinstance(scenarios, dict) else scenarios):  # ← FIX
#     csv_path = connectivity_results_path / f"catchment_connectivity__{key}__lambda_5.csv"
#     if not csv_path.exists():
#         print(f"WARNING: missing CSV → {csv_path}")
#         gdf_plot = catchments[["catchment_uid","geometry"]].copy()
#         gdf_plot["norm_scenario_pct"] = np.nan
#     else:
#         df = pd.read_csv(csv_path, usecols=["catchment_uid","norm_scenario_pct"])
#         gdf_plot = catchments[["catchment_uid","geometry"]].merge(df, on="catchment_uid", how="left")
#     gdfs.append((key, nice, gdf_plot))

# # Figure layout: 2 rows × 3 columns (last col = colorbar)
# fig = plt.figure(figsize=(16, 10), dpi=300)
# gs  = fig.add_gridspec(2, 3, width_ratios=[1, 1, 0.05], wspace=0.08, hspace=0.18)
# axes = [fig.add_subplot(gs[0,0]), fig.add_subplot(gs[0,1]),
#         fig.add_subplot(gs[1,0]), fig.add_subplot(gs[1,1])]
# cax  = fig.add_subplot(gs[:, 2])  # colorbar axis spans both rows

# norm = mpl.colors.Normalize(vmin=0, vmax=100)   # shared 0–100%
# cmap = plt.colormaps["Greens"]
# letters = ["a)", "b)", "c)", "d)"]

# for ax, letter, (key, nice, gdf_plot) in zip(axes, letters, gdfs):
#     try:
#         jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.6, zorder=1)
#     except Exception:
#         pass

#     gdf_plot.plot(
#         ax=ax, column="norm_scenario_pct", cmap=cmap, norm=norm,
#         linewidth=0.25, edgecolor="black", alpha=0.9, zorder=2,
#         missing_kwds={"color":"lightgrey","edgecolor":"white","hatch":"///","label":"No data"}
#     )

#     reps = gdf_plot.geometry.representative_point()
#     for (pt, uid) in zip(reps, gdf_plot["catchment_uid"]):
#         if pd.isna(uid):
#             continue
#         t = ax.text(pt.x, pt.y, str(int(uid)),
#                     ha="center", va="center", fontsize=6.5, fontweight="bold",
#                     color="black", zorder=3)
#         t.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])

#     ax.text(0.01, 0.99, letter, transform=ax.transAxes,
#             ha="left", va="top", fontsize=12, fontweight="bold",
#             path_effects=[pe.withStroke(linewidth=2, foreground="white")])
#     ax.set_title(f"{nice} (λ=5)", fontsize=12, pad=6)
#     ax.set_axis_off()

# # Shared colorbar on the dedicated axis (no overlap)
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
# cbar = fig.colorbar(sm, cax=cax)
# cbar.set_label("Normalized connectivity (%)")
# cbar.set_ticks([0, 20, 40, 60, 80, 100])

# # Put scale bar + north arrow on panel (b), top-right near the colorbar
# # Put scale bar + north arrow on panel (b), styled with YOUR helper defaults
# # (These use your linewidth/ticks/labels from the functions you pasted.)
# # Put scale bar + north arrow on panel (b), using your styled helpers
# add_scale_bar_rb(axes[1], length_km=20, location=(0.90, 0.80),
#                  linewidth=1, tick_height=0.012, label_offset=0.018, km_offset=0.010)
# add_north_arrow_rb(axes[1], location=(0.90, 0.87), size=0.055, fontsize=8, label_offset=0.015)
# panel_name = "Panel_normalized_connectivity_lambda5_2x2"
# out_png = Path(output_dir) / f"{panel_name}.png"
# out_pdf = Path(output_dir) / f"{panel_name}.pdf"
# fig.savefig(out_png, dpi=300, bbox_inches="tight")
# fig.savefig(out_pdf, bbox_inches="tight")
# print("Saved panel:", out_png.name, "and", out_pdf.name, "→", Path(output_dir).resolve())

# plt.show()

In [ ]:

# Scenario CSVs (λ=5)
paths = {
    "baseline":       baseline_connectivity_lambda_5_path,
    "partial":        partial_connectivity_lambda_5_path,
    "reforest_all":   reforest_all_connectivity_lambda_5_path,
    "long_timeframe": long_timeframe_connectivity_lambda_5_path,
}

# Load each CSV and rename the normalized column per scenario
dfs = []
for key, p in paths.items():
    df = pd.read_csv(p, usecols=["catchment_uid", "norm_scenario_pct"])
    df = df.rename(columns={"norm_scenario_pct": f"norm_{key}"})
    dfs.append(df)

# Merge all on catchment_uid
wide = reduce(lambda L, R: pd.merge(L, R, on="catchment_uid", how="outer"), dfs).sort_values("catchment_uid")

# Simple differences (percentage points)
wide["diff_partial_minus_baseline_pp"] = wide["norm_partial"]        - wide["norm_baseline"]
wide["diff_all_minus_partial_pp"]      = wide["norm_reforest_all"]   - wide["norm_partial"]
wide["diff_long_minus_all_pp"]         = wide["norm_long_timeframe"] - wide["norm_reforest_all"]

# Round selected columns to 2 decimals
cols_to_round = [
    "norm_baseline", "norm_partial", "norm_reforest_all", "norm_long_timeframe",
    "diff_partial_minus_baseline_pp", "diff_all_minus_partial_pp", "diff_long_minus_all_pp",
]
wide[cols_to_round] = wide[cols_to_round].round(2)

# Save + peek (force two decimals in CSV)
out_path = connectivity_results_path / "pairwise_norm_diffs__lambda_5.csv"
wide.to_csv(out_path, index=False, float_format="%.2f")
print("Saved:", out_path)

print(
    wide[["catchment_uid"] + cols_to_round]
        .head(10)
        .to_string(index=False)
)

In [ ]:
# # --- 3 stacked panels (no "reforest all") + right colorbar -------------------
# # (Assumes: scenarios, catchments, jamaica_boundary, connectivity_results_path,
# #           output_dir, add_scale_bar_rb, add_north_arrow_rb exist)


# # Load & merge attributes for each scenario (λ=5)
# gdfs = []
# iterable = scenarios.items() if isinstance(scenarios, dict) else scenarios
# for key, nice in iterable:
#     csv_path = connectivity_results_path / f"catchment_connectivity__{key}__lambda_5.csv"
#     if not csv_path.exists():
#         print(f"WARNING: missing CSV → {csv_path}")
#         gdf_plot = catchments[["catchment_uid","geometry"]].copy()
#         gdf_plot["norm_scenario_pct"] = pd.NA
#     else:
#         df = pd.read_csv(csv_path, usecols=["catchment_uid","norm_scenario_pct"])
#         gdf_plot = catchments[["catchment_uid","geometry"]].merge(df, on="catchment_uid", how="left")
#     gdfs.append((key, nice, gdf_plot))

# # Drop the "reforest all" scenario (robust to key or label variations)
# drop_terms = ("reforest_all", "reforest all")
# gdfs_filtered = [
#     (k, n, g) for (k, n, g) in gdfs
#     if not (any(t in str(k).lower() for t in drop_terms) or any(t in str(n).lower() for t in drop_terms))
# ]

# # Keep first 3 (preserving original order)
# if len(gdfs_filtered) < 3:
#     raise ValueError(f"Need at least 3 scenarios after filtering; got {len(gdfs_filtered)}.")
# gdfs3 = gdfs_filtered[:3]

# # Figure layout: 3 rows × 2 columns (last col = colorbar spans all rows)
# fig = plt.figure(figsize=(8, 12), dpi=300)
# gs  = fig.add_gridspec(3, 2, width_ratios=[1, 0.05], height_ratios=[1, 1, 1],
#                        wspace=0.06, hspace=0.16)
# axes = [fig.add_subplot(gs[0,0]), fig.add_subplot(gs[1,0]), fig.add_subplot(gs[2,0])]
# cax  = fig.add_subplot(gs[:, 1])  # colorbar axis spans all rows

# norm = mpl.colors.Normalize(vmin=0, vmax=100)   # shared 0–100%
# cmap = plt.colormaps["Greens"]
# letters = ["a)", "b)", "c)"]

# for ax, letter, (key, nice, gdf_plot) in zip(axes, letters, gdfs3):
#     try:
#         jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.6, zorder=1)
#     except Exception:
#         pass

#     gdf_plot.plot(
#         ax=ax, column="norm_scenario_pct", cmap=cmap, norm=norm,
#         linewidth=0.25, edgecolor="black", alpha=0.9, zorder=2,
#         missing_kwds={"color":"lightgrey","edgecolor":"white","hatch":"///","label":"No data"}
#     )

#     reps = gdf_plot.geometry.representative_point()
#     for (pt, uid) in zip(reps, gdf_plot["catchment_uid"]):
#         if pd.isna(uid):
#             continue
#         t = ax.text(pt.x, pt.y, str(int(uid)),
#                     ha="center", va="center", fontsize=6.5, fontweight="bold",
#                     color="black", zorder=3)
#         t.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])

#     ax.text(0.01, 0.99, letter, transform=ax.transAxes,
#             ha="left", va="top", fontsize=12, fontweight="bold",
#             path_effects=[pe.withStroke(linewidth=2, foreground="white")])
#     ax.set_title(f"{nice}", fontsize=12, pad=6)
#     ax.set_axis_off()

# # Shared colorbar on the dedicated axis
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
# cbar = fig.colorbar(sm, cax=cax)
# cbar.set_label("Normalized connectivity (%)")
# cbar.set_ticks([0, 20, 40, 60, 80, 100])

# # Scale bar + North arrow on the middle panel (b)
# try:
#     add_scale_bar_rb(axes[1], length_km=20, location=(0.90, 0.10),  # lower-right works better vertically
#                      linewidth=1, tick_height=0.012, label_offset=0.018, km_offset=0.010)
#     add_north_arrow_rb(axes[1], location=(0.90, 0.18), size=0.055, fontsize=8, label_offset=0.015)
# except Exception:
#     pass

# # Save
# panel_name = "Panel_normalized_connectivity_lambda5_3panel_vertical_no_reforest_all"
# out_png = Path(output_dir) / f"{panel_name}.png"
# out_pdf = Path(output_dir) / f"{panel_name}.pdf"
# fig.savefig(out_png, dpi=300, bbox_inches="tight")
# fig.savefig(out_pdf, bbox_inches="tight")
# print("Saved panel:", out_png.name, "and", out_pdf.name, "→", Path(output_dir).resolve())

# plt.show()

In [ ]:
# # --- 3 stacked panels (no "reforest all") + right colorbar -------------------
# # (Assumes: scenarios, catchments, jamaica_boundary, connectivity_results_path,
# #           output_dir, add_scale_bar_rb, add_north_arrow_rb exist)

# # Load & merge attributes for each scenario (input files are "...__lambda_5.csv";
# # titles will NOT display the λ)
# gdfs = []
# iterable = scenarios.items() if isinstance(scenarios, dict) else scenarios
# for key, nice in iterable:
#     csv_path = connectivity_results_path / f"catchment_connectivity__{key}__lambda_5.csv"
#     if not csv_path.exists():
#         print(f"WARNING: missing CSV → {csv_path}")
#         gdf_plot = catchments[["catchment_uid","geometry"]].copy()
#         gdf_plot["norm_scenario_pct"] = pd.NA
#     else:
#         df = pd.read_csv(csv_path, usecols=["catchment_uid","norm_scenario_pct"])
#         gdf_plot = catchments[["catchment_uid","geometry"]].merge(df, on="catchment_uid", how="left")
#     gdfs.append((key, nice, gdf_plot))

# # Drop the "reforest all" scenario (robust to key or label variations)
# drop_terms = ("reforest_all", "reforest all")
# gdfs_filtered = [
#     (k, n, g) for (k, n, g) in gdfs
#     if not (any(t in str(k).lower() for t in drop_terms) or any(t in str(n).lower() for t in drop_terms))
# ]

# # Keep first 3 (preserving original order)
# if len(gdfs_filtered) < 3:
#     raise ValueError(f"Need at least 3 scenarios after filtering; got {len(gdfs_filtered)}.")
# gdfs3 = gdfs_filtered[:3]

# # Rename the 2nd panel's title from whatever it was (e.g., "partial") to "restoration"
# k2, _, g2 = gdfs3[1]
# gdfs3[1] = (k2, "Restoration", g2)

# # Figure layout: 3 rows × 2 columns (last col = colorbar spans all rows)
# fig = plt.figure(figsize=(8, 12), dpi=300)
# gs  = fig.add_gridspec(3, 2, width_ratios=[1, 0.05], height_ratios=[1, 1, 1],
#                        wspace=0.06, hspace=0.16)
# axes = [fig.add_subplot(gs[0,0]), fig.add_subplot(gs[1,0]), fig.add_subplot(gs[2,0])]
# cax  = fig.add_subplot(gs[:, 1])  # colorbar axis spans all rows

# norm = mpl.colors.Normalize(vmin=0, vmax=100)   # shared 0–100%
# cmap = plt.colormaps["Greens"]
# letters = ["a)", "b)", "c)"]

# for ax, letter, (key, nice, gdf_plot) in zip(axes, letters, gdfs3):
#     try:
#         jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.6, zorder=1)
#     except Exception:
#         pass

#     gdf_plot.plot(
#         ax=ax, column="norm_scenario_pct", cmap=cmap, norm=norm,
#         linewidth=0.25, edgecolor="black", alpha=0.9, zorder=2,
#         missing_kwds={"color":"lightgrey","edgecolor":"white","hatch":"///","label":"No data"}
#     )

#     reps = gdf_plot.geometry.representative_point()
#     for (pt, uid) in zip(reps, gdf_plot["catchment_uid"]):
#         if pd.isna(uid):
#             continue
#         t = ax.text(pt.x, pt.y, str(int(uid)),
#                     ha="center", va="center", fontsize=6.5, fontweight="bold",
#                     color="black", zorder=3)
#         t.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])

#     ax.text(0.01, 0.99, letter, transform=ax.transAxes,
#             ha="left", va="top", fontsize=12, fontweight="bold",
#             path_effects=[pe.withStroke(linewidth=2, foreground="white")])
#     ax.set_title(f"{nice}", fontsize=12, pad=6)  # no "(λ=5)"
#     ax.set_axis_off()

# # Shared colorbar on the dedicated axis
# sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
# cbar = fig.colorbar(sm, cax=cax)
# cbar.set_label("Normalized connectivity (%)")
# cbar.set_ticks([0, 20, 40, 60, 80, 100])

# # Scale bar + North arrow on the middle panel (b)
# try:
#     add_scale_bar_rb(axes[1], length_km=20, location=(0.90, 0.10),
#                      linewidth=1, tick_height=0.012, label_offset=0.018, km_offset=0.010)
#     add_north_arrow_rb(axes[1], location=(0.90, 0.18), size=0.055, fontsize=8, label_offset=0.015)
# except Exception:
#     pass

# # Save (filename without "lambda5")
# panel_name = "Panel_normalized_connectivity_3panel_vertical_no_reforest_all"
# out_png = Path(output_dir) / f"{panel_name}.png"
# out_pdf = Path(output_dir) / f"{panel_name}.pdf"
# fig.savefig(out_png, dpi=300, bbox_inches="tight")
# fig.savefig(out_pdf, bbox_inches="tight")
# print("Saved panel:", out_png.name, "and", out_pdf.name, "→", Path(output_dir).resolve())

# plt.show()

In [ ]:
# --- 3 stacked panels (no "reforest all") + right colorbar -------------------
# (Assumes: scenarios, catchments, jamaica_boundary, connectivity_results_path,
#           output_dir, add_scale_bar_rb, add_north_arrow_rb exist)

# Load & merge attributes for each scenario (input files are "...__lambda_5.csv";
# titles will NOT display the λ)
gdfs = []
iterable = scenarios.items() if isinstance(scenarios, dict) else scenarios
for key, nice in iterable:
    csv_path = connectivity_results_path / f"catchment_connectivity__{key}__lambda_5.csv"
    if not csv_path.exists():
        print(f"WARNING: missing CSV → {csv_path}")
        gdf_plot = catchments[["catchment_uid","geometry"]].copy()
        gdf_plot["norm_scenario_pct"] = pd.NA
    else:
        df = pd.read_csv(csv_path, usecols=["catchment_uid","norm_scenario_pct"])
        gdf_plot = catchments[["catchment_uid","geometry"]].merge(df, on="catchment_uid", how="left")
    gdfs.append((key, nice, gdf_plot))

# Drop the "reforest all" scenario (robust to key or label variations)
drop_terms = ("reforest_all", "reforest all")
gdfs_filtered = [
    (k, n, g) for (k, n, g) in gdfs
    if not (any(t in str(k).lower() for t in drop_terms) or any(t in str(n).lower() for t in drop_terms))
]

# Keep first 3 (preserving original order)
if len(gdfs_filtered) < 3:
    raise ValueError(f"Need at least 3 scenarios after filtering; got {len(gdfs_filtered)}.")
gdfs3 = gdfs_filtered[:3]

# Rename the 2nd panel's title to "Restoration"
k2, _, g2 = gdfs3[1]
gdfs3[1] = (k2, "Restoration", g2)

# Figure layout: 3 rows × 2 columns (last col = colorbar spans all rows)
fig = plt.figure(figsize=(8, 12), dpi=300)
gs  = fig.add_gridspec(3, 2, width_ratios=[1, 0.05], height_ratios=[1, 1, 1],
                       wspace=0.06, hspace=0.16)
axes = [fig.add_subplot(gs[0,0]), fig.add_subplot(gs[1,0]), fig.add_subplot(gs[2,0])]
cax  = fig.add_subplot(gs[:, 1])  # colorbar axis spans all rows

norm = mpl.colors.Normalize(vmin=0, vmax=100)   # shared 0–100%
cmap = plt.colormaps["Greens"]
letters = ["a)", "b)", "c)"]

for ax, letter, (key, nice, gdf_plot) in zip(axes, letters, gdfs3):
    try:
        jamaica_boundary.boundary.plot(ax=ax, color="black", linewidth=0.6, zorder=1)
    except Exception:
        pass

    gdf_plot.plot(
        ax=ax, column="norm_scenario_pct", cmap=cmap, norm=norm,
        linewidth=0.25, edgecolor="black", alpha=0.9, zorder=2,
        missing_kwds={"color":"lightgrey","edgecolor":"white","hatch":"///","label":"No data"}
    )

    reps = gdf_plot.geometry.representative_point()
    for (pt, uid) in zip(reps, gdf_plot["catchment_uid"]):
        if pd.isna(uid):
            continue
        t = ax.text(pt.x, pt.y, str(int(uid)),
                    ha="center", va="center", fontsize=6.5, fontweight="bold",
                    color="black", zorder=3)
        t.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])

    ax.text(0.01, 0.99, letter, transform=ax.transAxes,
            ha="left", va="top", fontsize=12, fontweight="bold",
            path_effects=[pe.withStroke(linewidth=2, foreground="white")])
    ax.set_title(f"{nice}", fontsize=12, pad=6)  # no "(λ=5)"
    ax.set_axis_off()

# Shared colorbar on the dedicated axis
sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Normalized connectivity (%)")
cbar.set_ticks([0, 20, 40, 60, 80, 100])

# North arrow (top-right) + Scale bar (just below it) on the TOP panel (a)
try:
    add_north_arrow_rb(
        axes[0],
        location=(0.92, 0.92),   # top-right in axes coords
        size=0.055,
        fontsize=8,
        label_offset=0.015
    )
    add_scale_bar_rb(
        axes[0],
        length_km=20,
        location=(0.92, 0.84),   # slightly below the arrow
        linewidth=1,
        tick_height=0.012,
        label_offset=0.018,
        km_offset=0.010
    )
except Exception:
    pass

# Save (filename without "lambda5")
panel_name = "Panel_normalized_connectivity_3panel_vertical_no_reforest_all"
out_png = Path(output_dir) / f"{panel_name}.png"
out_pdf = Path(output_dir) / f"{panel_name}.pdf"
fig.savefig(out_png, dpi=300, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
print("Saved panel:", out_png.name, "and", out_pdf.name, "→", Path(output_dir).resolve())

plt.show()